https://github.com/Niv-Kor/Target-Score-Detector?tab=readme-ov-file

In [ ]:
import cv2
import numpy as np
from tkinter import Tk, filedialog

# === 1️⃣ Pedir imagen al usuario ===
Tk().withdraw()  # Ocultar ventana principal de Tkinter
ruta_imagen = filedialog.askopenfilename(
    title="Selecciona una imagen de la diana",
    filetypes=[("Imágenes", "*.jpg *.png *.jpeg *.bmp")]
)

if not ruta_imagen:
    print("No se seleccionó ninguna imagen. Saliendo...")
    exit()

# === 2️⃣ Cargar imagen ===
img = cv2.imread(ruta_imagen)
if img is None:
    print("Error al cargar la imagen.")
    exit()

# --- parámetros ajustables ---
MIN_AREA = 120  # área mínima para considerar un impacto

# === 3️⃣ Procesamiento de imagen ===
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# Umbral binario inverso (para impactos oscuros sobre fondo claro)
_, thresh = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)

# Limpiar pequeñas imperfecciones
kernel = np.ones((3, 3), np.uint8)
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

# === 4️⃣ Encontrar contornos (posibles disparos) ===
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# === 5️⃣ Calcular centro de la diana (centro de la imagen) ===
h, w = img.shape[:2]
centro = (w // 2, h // 2)

# Dibujar el centro
cv2.circle(img, centro, 6, (255, 255, 0), -1)
cv2.putText(img, "Centro", (centro[0] + 10, centro[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

# === 6️⃣ Analizar disparos ===
disparos = []
for i, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    if area > MIN_AREA:
        (x, y), radius = cv2.minEnclosingCircle(cnt)
        cX, cY = int(x), int(y)
        distancia = np.hypot(cX - centro[0], cY - centro[1])
        max_radio = min(w, h) // 2

        # Escalar puntuación: 10 puntos en el centro → 0 en el borde
        puntuacion = max(0, int(10 - (distancia / max_radio) * 10))
        disparos.append((cX, cY, puntuacion, int(distancia)))

        # Dibujar impacto
        cv2.circle(img, (cX, cY), int(radius), (0, 255, 0), 2)
        cv2.circle(img, (cX, cY), 4, (0, 0, 255), -1)
        cv2.putText(img, f"#{i+1} ({puntuacion})", (cX + 10, cY - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

# === 7️⃣ Mostrar resultados en pantalla ===
total_puntos = sum([p for _, _, p, _ in disparos])
num_disparos = len(disparos)

# Cuadro resumen
cv2.rectangle(img, (20, 20), (350, 120), (0, 0, 0), -1)
cv2.putText(img, f"Disparos: {num_disparos}", (30, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
cv2.putText(img, f"Total puntos: {total_puntos}", (30, 100),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

# === 8️⃣ Mostrar y guardar resultado ===
cv2.imshow("Disparos detectados", img)
print(f"\n🔍 Disparos detectados: {num_disparos}")
for i, (x, y, p, d) in enumerate(disparos, start=1):
    print(f"  • Disparo {i}: posición=({x},{y}), distancia={d}px, puntuación={p}")

print(f"\n🎯 Puntuación total: {total_puntos} puntos")

cv2.waitKey(0)
cv2.destroyAllWindows()


No se seleccionó ninguna imagen. Saliendo...
Error al cargar la imagen.


error: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:196: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


: 

# Detección en vivo através de una webcam


In [ ]:
import cv2
import numpy as np

# --- Parámetros ajustables ---
MIN_AREA = 120  # área mínima para considerar un impacto

# === 1️⃣ Abrir webcam ===
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("No se pudo acceder a la cámara.")
    exit()

print("🎯 Detector en vivo iniciado. Pulsa 'q' para salir.")

while True:
    ret, img = cap.read()
    if not ret:
        break

    # === 2️⃣ Procesamiento de imagen ===
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)

    kernel = np.ones((3, 3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # === 3️⃣ Centro de la diana (centro del frame) ===
    h, w = img.shape[:2]
    centro = (w // 2, h // 2)
    cv2.circle(img, centro, 6, (255, 255, 0), -1)
    cv2.putText(img, "Centro", (centro[0] + 10, centro[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    # === 4️⃣ Analizar disparos ===
    disparos = []
    for i, cnt in enumerate(contours):
        area = cv2.contourArea(cnt)
        if area > MIN_AREA:
            (x, y), radius = cv2.minEnclosingCircle(cnt)
            cX, cY = int(x), int(y)
            distancia = np.hypot(cX - centro[0], cY - centro[1])
            max_radio = min(w, h) // 2
            puntuacion = max(0, int(10 - (distancia / max_radio) * 10))
            disparos.append((cX, cY, puntuacion))

            cv2.circle(img, (cX, cY), int(radius), (0, 255, 0), 2)
            cv2.circle(img, (cX, cY), 4, (0, 0, 255), -1)
            cv2.putText(img, f"{puntuacion}", (cX + 10, cY - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    num_disparos = len(disparos)
    total_puntos = sum([p for _, _, p in disparos])

    # === 5️⃣ Mostrar resultados ===
    cv2.rectangle(img, (20, 20), (350, 120), (0, 0, 0), -1)
    cv2.putText(img, f"Disparos: {num_disparos}", (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(img, f"Total puntos: {total_puntos}", (30, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.imshow("Deteccion en vivo de Disparos", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


# Versión avanzada (detectar limites de la Diana):


1️⃣ Detectar el círculo de la diana
cv2.HoughCircles
Detecta el borde circular más grande (la diana física)

2️⃣ Calcular el centro automáticamente
Centro del círculo detectado
Evita depender del centro del frame


3️⃣ Detectar los disparos
Umbral por oscuridad + contornos
Los impactos suelen ser zonas negras pequeñas dentro de la diana



El temblor hace que la detección de la diana y los disparos varíe constantemente. 

In [ ]:
import cv2
import numpy as np

# --- Parámetros ajustables ---
MIN_AREA_DISPARO = 80   # área mínima para considerar impacto
PARAM_CANNY = 100       # sensibilidad para detección de bordes
BLUR = 5                # suavizado
DEBUG = True            # muestra información extra

# === 1️⃣ Abrir webcam ===
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("No se pudo acceder a la cámara.")
    exit()

print("🎯 Detector de diana en vivo iniciado. Pulsa 'q' para salir.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = cv2.flip(frame, 1)  # espejo para más comodidad
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (BLUR, BLUR), 0)

    # === 2️⃣ Detectar círculo grande (la diana) ===
    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=100,
        param1=PARAM_CANNY,
        param2=50,
        minRadius=100,
        maxRadius=0
    )

    centro_diana = None
    radio_diana = None

    if circles is not None:
        circles = np.uint16(np.around(circles))
        # Tomamos el círculo más grande
        circles = sorted(circles[0, :], key=lambda c: c[2], reverse=True)
        (x, y, r) = circles[0]
        centro_diana = (x, y)
        radio_diana = r

        cv2.circle(img, (x, y), r, (255, 255, 0), 2)
        cv2.circle(img, (x, y), 5, (0, 255, 255), -1)
        cv2.putText(img, "Centro diana", (x + 10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    # === 3️⃣ Detectar impactos (zonas oscuras) ===
    _, thresh = cv2.threshold(blur, 60, 255, cv2.THRESH_BINARY_INV)
    kernel = np.ones((3, 3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    disparos = []

    if centro_diana is not None:
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area > MIN_AREA_DISPARO:
                (x, y), radius = cv2.minEnclosingCircle(cnt)
                cX, cY = int(x), int(y)

                # Comprobamos que esté dentro de la diana
                distancia = np.hypot(cX - centro_diana[0], cY - centro_diana[1])
                if distancia < radio_diana:
                    max_radio = radio_diana
                    puntuacion = max(0, int(10 - (distancia / max_radio) * 10))
                    disparos.append((cX, cY, puntuacion))

                    cv2.circle(img, (cX, cY), int(radius), (0, 255, 0), 2)
                    cv2.putText(img, f"{puntuacion}", (cX + 10, cY - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    # === 4️⃣ Mostrar resultados ===
    num_disparos = len(disparos)
    total_puntos = sum([p for _, _, p in disparos])

    overlay = img.copy()
    cv2.rectangle(overlay, (20, 20), (350, 120), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, img, 0.5, 0, img)

    cv2.putText(img, f"Disparos: {num_disparos}", (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(img, f"Total puntos: {total_puntos}", (30, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    if DEBUG:
        cv2.imshow("Umbral disparos", thresh)

    cv2.imshow("🎯 Detección de Disparos", img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


🎯 Detector de diana en vivo iniciado. Pulsa 'q' para salir.


C:\Users\visionado\AppData\Local\Temp\ipykernel_7820\3568419712.py:71: RuntimeWarning: overflow encountered in scalar subtract
  distancia = np.hypot(cX - centro_diana[0], cY - centro_diana[1])


# Con estabilizador

In [1]:
import cv2
import numpy as np

# ---------------- Ajustables ----------------
MIN_AREA_BASE = 60        # área mínima base para blob
BILATERAL_D = 9
CLAHE_CLIP = 2.0
CLAHE_GRID = (8, 8)
DP = 1.2
CANNY_PARAM1 = 100
HOUGH_PARAM2 = 40
MIN_CONFIRM = 3           # cuántos frames seguidos para confirmar un impacto
MAX_LOST = 6              # cuántos frames sin ver un objeto antes de borrarlo
DIST_THRESH = 40          # distancia máxima (px) para emparejar detecciones con objetos
# --------------------------------------------

class CentroidTracker:
    def __init__(self, max_lost=6, min_confirmed=3, dist_thresh=40):
        self.nextID = 0
        self.objects = {}  # id -> {'centroid':(x,y), 'frames_seen':int, 'frames_missing':int, 'confirmed':bool}
        self.max_lost = max_lost
        self.min_confirmed = min_confirmed
        self.dist_thresh = dist_thresh

    def register(self, centroid):
        self.objects[self.nextID] = {'centroid': centroid, 'frames_seen': 1, 'frames_missing': 0, 'confirmed': False}
        self.nextID += 1
        return self.nextID - 1

    def deregister(self, objectID):
        del self.objects[objectID]

    def update(self, detections):
        """
        detections: list of (x,y)
        devuelve lista de objectIDs que se acaban de confirmar.
        """
        newly_confirmed = []
        if len(detections) == 0:
            for oid in list(self.objects.keys()):
                self.objects[oid]['frames_missing'] += 1
                if self.objects[oid]['frames_missing'] > self.max_lost:
                    self.deregister(oid)
            return newly_confirmed

        if len(self.objects) == 0:
            for det in detections:
                self.register(det)
            return newly_confirmed

        objectIDs = list(self.objects.keys())
        objectCentroids = [self.objects[oid]['centroid'] for oid in objectIDs]

        D = np.linalg.norm(np.array(objectCentroids)[:, None, :] - np.array(detections)[None, :, :], axis=2)
        assigned_rows, assigned_cols = set(), set()

        # emparejado greedy por mínima distancia mientras la distancia < umbral
        while True:
            min_idx = np.unravel_index(np.argmin(D, axis=None), D.shape)
            r, c = min_idx
            if D[r, c] > self.dist_thresh:
                break
            oid = objectIDs[r]
            det = detections[c]
            old = np.array(self.objects[oid]['centroid'])
            new = np.array(det)
            # suavizado del centro (evita saltos bruscos)
            self.objects[oid]['centroid'] = tuple((0.6 * old + 0.4 * new).astype(int))
            self.objects[oid]['frames_seen'] += 1
            self.objects[oid]['frames_missing'] = 0
            if (not self.objects[oid]['confirmed']) and (self.objects[oid]['frames_seen'] >= self.min_confirmed):
                self.objects[oid]['confirmed'] = True
                newly_confirmed.append(oid)
            assigned_rows.add(r); assigned_cols.add(c)
            D[r, :] = np.inf
            D[:, c] = np.inf
            if np.isinf(D).all():
                break

        # detecciones nuevas
        for c in range(D.shape[1]):
            if c not in assigned_cols:
                self.register(tuple(detections[c]))

        # objetos no emparejados -> incrementar missing
        for r in range(D.shape[0]):
            if r not in assigned_rows:
                oid = objectIDs[r]
                self.objects[oid]['frames_missing'] += 1
                if self.objects[oid]['frames_missing'] > self.max_lost:
                    self.deregister(oid)

        return newly_confirmed

def make_blob_detector(min_area=40, max_area=5000):
    params = cv2.SimpleBlobDetector_Params()
    params.filterByColor = True
    params.blobColor = 0  # buscamos blobs oscuros
    params.filterByArea = True
    params.minArea = max(5, int(min_area))
    params.maxArea = max_area
    params.filterByCircularity = True
    params.minCircularity = 0.4
    params.filterByInertia = True
    params.minInertiaRatio = 0.2
    params.filterByConvexity = True
    params.minConvexity = 0.6
    return cv2.SimpleBlobDetector_create(params)

# --------------- Main ---------------
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("No se pudo abrir la cámara"); exit()

ct = CentroidTracker(max_lost=MAX_LOST, min_confirmed=MIN_CONFIRM, dist_thresh=DIST_THRESH)
blob_detector = make_blob_detector(MIN_AREA_BASE, 5000)
last_circle = None
last_circle_age = 0
confirmed_scores = {}  # id -> score

while True:
    ret, frame = cap.read()
    if not ret: break
    img = cv2.flip(frame, 1)
    h, w = img.shape[:2]

    # PREPROCESADO: CLAHE + bilateral + blur
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
    gray = clahe.apply(gray)
    gray = cv2.bilateralFilter(gray, BILATERAL_D, 75, 75)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # HOUGH para detectar la diana (círculo más grande)
    minR = int(min(h, w) * 0.15)
    maxR = int(min(h, w) * 0.5)
    circles = cv2.HoughCircles(blur, cv2.HOUGH_GRADIENT, dp=DP, minDist=100,
                               param1=CANNY_PARAM1, param2=HOUGH_PARAM2,
                               minRadius=minR, maxRadius=maxR)
    if circles is not None:
        circles = np.uint16(np.around(circles[0]))
        circles = sorted(circles, key=lambda c: c[2], reverse=True)
        x, y, r = circles[0]
        last_circle = (x, y, r)
        last_circle_age = 0
    else:
        last_circle_age += 1
        # fallback: usar la última diana conocida hasta N frames
        if last_circle_age > 30:
            last_circle = None

    # mascara para limitar búsqueda dentro de la diana
    mask = np.zeros_like(gray)
    if last_circle is not None:
        cx, cy, cr = last_circle
        cv2.circle(mask, (cx, cy), cr - 2, 255, -1)
    masked = cv2.bitwise_and(gray, gray, mask=mask)

    # UMBRALIZADO de impactos oscuros: Otsu en ROI (si hay diana)
    if last_circle is not None:
        x1 = max(0, cx - cr); x2 = min(w, cx + cr)
        y1 = max(0, cy - cr); y2 = min(h, cy + cr)
        roi = masked[y1:y2, x1:x2]
        if roi.size == 0:
            _, thresh_full = cv2.threshold(masked, 60, 255, cv2.THRESH_BINARY_INV)
        else:
            _, thresh_roi = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
            thresh_roi = cv2.morphologyEx(thresh_roi, cv2.MORPH_OPEN, kernel, iterations=1)
            thresh_roi = cv2.morphologyEx(thresh_roi, cv2.MORPH_CLOSE, kernel, iterations=2)
            thresh_full = np.zeros_like(masked)
            thresh_full[y1:y2, x1:x2] = thresh_roi
    else:
        _, thresh_full = cv2.threshold(masked, 60, 255, cv2.THRESH_BINARY_INV)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        thresh_full = cv2.morphologyEx(thresh_full, cv2.MORPH_OPEN, kernel, iterations=1)

    # Si hay diana, ajustar min/max area relativo al radio
    if last_circle is not None:
        min_area = max(MIN_AREA_BASE, int(np.pi * (cr * 0.01) ** 2))
        max_area = int(np.pi * (cr * 0.2) ** 2)
        blob_detector = make_blob_detector(min_area, max_area)

    keypoints = blob_detector.detect(thresh_full)
    detections = []
    for kp in keypoints:
        xk = int(kp.pt[0]); yk = int(kp.pt[1])
        # comprobar dentro de la diana si se conoce
        if last_circle is None or np.hypot(xk - cx, yk - cy) <= cr:
            detections.append((xk, yk))

    # actualizar tracker: detecciones -> objetos con IDs
    new_confirmed = ct.update(detections)

    # cuando un objeto se confirma, calculamos y guardamos su puntuación (una vez)
    for oid in new_confirmed:
        centroid = ct.objects[oid]['centroid']
        if last_circle is not None:
            dist = np.hypot(centroid[0] - cx, centroid[1] - cy)
            score = max(0, int(10 - (dist / cr) * 10))
        else:
            score = 0
        confirmed_scores[oid] = score

    # contadores
    visible_count = len(detections)
    visible_points = 0
    if last_circle is not None:
        for d in detections:
            dist = np.hypot(d[0] - cx, d[1] - cy)
            visible_points += max(0, int(10 - (dist / cr) * 10))
    confirmed_count = len(confirmed_scores)
    confirmed_points = sum(confirmed_scores.values())

    # DIBUJOS y overlay semitransparente
    overlay = img.copy()
    if last_circle is not None:
        cv2.circle(img, (cx, cy), cr, (255, 255, 0), 2)
        cv2.circle(img, (cx, cy), 4, (0, 255, 255), -1)
    for kp in keypoints:
        xk = int(kp.pt[0]); yk = int(kp.pt[1]); rad = int(kp.size / 2)
        cv2.circle(img, (xk, yk), rad, (0, 255, 0), 2)
    for oid, data in ct.objects.items():
        c = data['centroid']
        color = (0, 200, 255) if data['confirmed'] else (200, 200, 0)
        cv2.circle(img, tuple(c), 6, color, -1)
        cv2.putText(img, f"ID{oid}", (c[0] + 8, c[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.rectangle(overlay, (10, 10), (380, 140), (0, 0, 0), -1)
    alpha = 0.5
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)

    cv2.putText(img, f"Visibles: {visible_count}  Pts visibles: {visible_points}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.putText(img, f"Confirmados: {confirmed_count}  Pts confirmados: {confirmed_points}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(img, "Pulsa q para salir", (20, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

    cv2.imshow("Thresh", thresh_full)
    cv2.imshow("Deteccion estable", img)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


C:\Users\visionado\AppData\Local\Temp\ipykernel_7348\875469425.py:161: RuntimeWarning: overflow encountered in scalar subtract
  x1 = max(0, cx - cr); x2 = min(w, cx + cr)
C:\Users\visionado\AppData\Local\Temp\ipykernel_7348\875469425.py:162: RuntimeWarning: overflow encountered in scalar subtract
  y1 = max(0, cy - cr); y2 = min(h, cy + cr)
C:\Users\visionado\AppData\Local\Temp\ipykernel_7348\875469425.py:189: RuntimeWarning: overflow encountered in scalar subtract
  if last_circle is None or np.hypot(xk - cx, yk - cy) <= cr:


KeyboardInterrupt: 